In [35]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact
import info
import loader
import stats
import plotter

dataset_path = "../dataset/processed/features.csv"
    
try:
    df = loader.load_data(dataset_path)
except FileNotFoundError as e:
    print(f"Error: dataset file ({dataset_path}) not found")
    sys.exit(1)

### Contare lingue per valore di una feature

In [36]:
qdf = df.copy()

def select_feature(features):
  feature_id = qdf[qdf["Parameter_Name"] == features]["Parameter_ID"].iloc[0]

  q2df = stats.filter(qdf, "Parameter_ID", [feature_id])
  q2df = q2df.groupby("Code_Name")["Language_ID"].nunique().sort_values(ascending=False)

  plotter.bar_plot(
    q2df,
    title="Numero di lingue per dimensione dell'inventario fonetico",
  )

  plotter.pie_plot(
    q2df,
    title="Numero di lingue per dimensione dell'inventario fonetico",
  )

interact(select_feature, features=sorted(qdf["Parameter_Name"].unique().tolist()));

interactive(children=(Dropdown(description='features', options=("'Want' Complement Subjects", "'When' Clauses"…

### Conto lingue per valore di feature divise in famiglia linguistica

In [37]:
qdf = df.copy()

def select_feature(features):
  feature_id = qdf[qdf["Parameter_Name"] == features]["Parameter_ID"].iloc[0]
  
  q2df = stats.filter(qdf, "Parameter_ID", [feature_id])
  q2df = pd.crosstab(q2df["Family"], q2df["Code_Name"])

  n_top = 40
  q2df = q2df.assign(Total=q2df.sum(axis=1))


  q2df = q2df.drop(index="other", errors="ignore")

  languages_by_family = q2df.sort_values("Total", ascending=False)
  languages_by_family = languages_by_family.drop("Total", axis=1)

  chunk = stats.get_chunks(languages_by_family, 40)[0]

  plotter.stacked_bar_plot(chunk)

interact(select_feature, features=sorted(qdf["Parameter_Name"].unique().tolist()));

interactive(children=(Dropdown(description='features', options=("'Want' Complement Subjects", "'When' Clauses"…

In [38]:
qdf = df.copy()

def select_family_and_feature(features, families):

    rows = qdf[qdf["Parameter_Name"] == features]

    # No feature for that family, i.e. empty result
    if (rows.empty):
      return

    feature_id = rows["Parameter_ID"].iloc[0]
     
    q2df = stats.filter(qdf, "Family", [families])
    q2df = stats.filter(q2df, "Parameter_ID", [feature_id])
    q2df = q2df.groupby("Code_Name")["Language_ID"].nunique().sort_values(ascending=False)

    plotter.pie_plot(
      q2df,
      figsize=(10, 10),
      title="Numero di lingue per feature all'interno di una particolare famiglia"
    )

interact(
    select_family_and_feature, 
    features=sorted(qdf["Parameter_Name"].unique().tolist()),
    families=sorted(qdf["Family"].unique().tolist())
);

interactive(children=(Dropdown(description='features', options=("'Want' Complement Subjects", "'When' Clauses"…